# 01 - Baselines

**Objectif** : Entrainer les modeles baselines et evaluer leurs performances sur le test set.

## Modeles
1. TF-IDF + Logistic Regression (OneVsRest)
2. TF-IDF + Linear SVM (OneVsRest)
3. TF-IDF + Multinomial Naive Bayes (OneVsRest)
4. BERT (bert-base-uncased) fine-tune -- baseline deep learning

## Metriques
- ROC-AUC (macro)
- F1-score (macro)
- Precision / Recall (macro)
- Hamming Loss
- Temps d'entrainement / inference

In [ ]:
import pandas as pd
import numpy as np
import time
import json
import os
import warnings
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.multiclass import OneVsRestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score, hamming_loss,
    classification_report
)

warnings.filterwarnings('ignore')

LABEL_COLS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
DATA_DIR = '../data/processed/'
MODELS_DIR = '../artifacts/models/'
METRICS_DIR = '../artifacts/metrics/'

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

## 1. Chargement des donnees

In [ ]:
train_df = pd.read_csv(DATA_DIR + 'train.csv')
val_df = pd.read_csv(DATA_DIR + 'val.csv')
test_df = pd.read_csv(DATA_DIR + 'test.csv')

print(f"Train : {len(train_df)}")
print(f"Val   : {len(val_df)}")
print(f"Test  : {len(test_df)}")

X_train = train_df['comment_text'].values
X_val = val_df['comment_text'].values
X_test = test_df['comment_text'].values

y_train = train_df[LABEL_COLS].values
y_val = val_df[LABEL_COLS].values
y_test = test_df[LABEL_COLS].values

## 2. TF-IDF Vectorization

In [ ]:
print("Fitting TF-IDF...")
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    strip_accents='unicode'
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)
X_test_tfidf = tfidf.transform(X_test)

print(f"TF-IDF shape : {X_train_tfidf.shape}")

## 3. Fonctions utilitaires

In [ ]:
def evaluate_model(y_true, y_pred, y_prob, model_name, train_time, inference_time):
    """Evalue un modele et retourne un dictionnaire de metriques."""
    metrics = {
        'model': model_name,
        'roc_auc_macro': roc_auc_score(y_true, y_prob, average='macro'),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'hamming_loss': hamming_loss(y_true, y_pred),
        'train_time_sec': round(train_time, 2),
        'inference_time_sec': round(inference_time, 4),
    }

    # Metriques par label
    for i, label in enumerate(LABEL_COLS):
        metrics[f'roc_auc_{label}'] = roc_auc_score(y_true[:, i], y_prob[:, i])

    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    print(f"  ROC-AUC (macro) : {metrics['roc_auc_macro']:.4f}")
    print(f"  F1 (macro)      : {metrics['f1_macro']:.4f}")
    print(f"  Precision       : {metrics['precision_macro']:.4f}")
    print(f"  Recall          : {metrics['recall_macro']:.4f}")
    print(f"  Hamming Loss    : {metrics['hamming_loss']:.4f}")
    print(f"  Train time      : {metrics['train_time_sec']:.2f}s")
    print(f"  Inference time  : {metrics['inference_time_sec']:.4f}s")

    return metrics


all_results = []

## 4. Baseline 1 : TF-IDF + Logistic Regression

In [ ]:
model_name = 'TF-IDF + Logistic Regression'

clf_lr = OneVsRestClassifier(LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs'))

t0 = time.time()
clf_lr.fit(X_train_tfidf, y_train)
train_time = time.time() - t0

t0 = time.time()
y_pred_lr = clf_lr.predict(X_test_tfidf)
y_prob_lr = clf_lr.predict_proba(X_test_tfidf)
inference_time = time.time() - t0

results_lr = evaluate_model(y_test, y_pred_lr, y_prob_lr, model_name, train_time, inference_time)
all_results.append(results_lr)

joblib.dump(clf_lr, MODELS_DIR + 'baseline_tfidf_logreg.joblib')
print(f"\nModele sauvegarde dans {MODELS_DIR}baseline_tfidf_logreg.joblib")

## 5. Baseline 2 : TF-IDF + Linear SVM

In [ ]:
model_name = 'TF-IDF + Linear SVM'

# LinearSVC n'a pas de predict_proba, on utilise CalibratedClassifierCV
clf_svm = OneVsRestClassifier(
    CalibratedClassifierCV(LinearSVC(max_iter=2000, C=1.0), cv=3)
)

t0 = time.time()
clf_svm.fit(X_train_tfidf, y_train)
train_time = time.time() - t0

t0 = time.time()
y_pred_svm = clf_svm.predict(X_test_tfidf)
y_prob_svm = clf_svm.predict_proba(X_test_tfidf)
inference_time = time.time() - t0

results_svm = evaluate_model(y_test, y_pred_svm, y_prob_svm, model_name, train_time, inference_time)
all_results.append(results_svm)

joblib.dump(clf_svm, MODELS_DIR + 'baseline_tfidf_svm.joblib')
print(f"\nModele sauvegarde dans {MODELS_DIR}baseline_tfidf_svm.joblib")

## 6. Baseline 3 : TF-IDF + Multinomial Naive Bayes

In [ ]:
model_name = 'TF-IDF + Naive Bayes'

clf_nb = OneVsRestClassifier(MultinomialNB(alpha=0.1))

t0 = time.time()
clf_nb.fit(X_train_tfidf, y_train)
train_time = time.time() - t0

t0 = time.time()
y_pred_nb = clf_nb.predict(X_test_tfidf)
y_prob_nb = clf_nb.predict_proba(X_test_tfidf)
inference_time = time.time() - t0

results_nb = evaluate_model(y_test, y_pred_nb, y_prob_nb, model_name, train_time, inference_time)
all_results.append(results_nb)

joblib.dump(clf_nb, MODELS_DIR + 'baseline_tfidf_nb.joblib')
print(f"\nModele sauvegarde dans {MODELS_DIR}baseline_tfidf_nb.joblib")

## 7. Baseline 4 : BERT (bert-base-uncased) fine-tune

Baseline deep learning de reference. On fine-tune BERT avec Hugging Face Transformers.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from sklearn.metrics import roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else
                       'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device : {device}")

In [ ]:
class ToxicDataset(Dataset):
    """Dataset PyTorch pour la classification multi-label."""

    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

In [ ]:
def compute_metrics_hf(eval_pred):
    """Fonction de metriques pour le Trainer HuggingFace."""
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs > 0.5).astype(int)

    return {
        'roc_auc_macro': roc_auc_score(labels, probs, average='macro'),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
    }

In [ ]:
MODEL_NAME_BERT = 'bert-base-uncased'

tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_NAME_BERT)

train_dataset = ToxicDataset(X_train, y_train, tokenizer_bert, max_length=256)
val_dataset = ToxicDataset(X_val, y_val, tokenizer_bert, max_length=256)
test_dataset = ToxicDataset(X_test, y_test, tokenizer_bert, max_length=256)

print(f"Train dataset : {len(train_dataset)}")
print(f"Val dataset   : {len(val_dataset)}")
print(f"Test dataset  : {len(test_dataset)}")

In [ ]:
model_bert = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_BERT,
    num_labels=len(LABEL_COLS),
    problem_type='multi_label_classification'
)

training_args = TrainingArguments(
    output_dir=MODELS_DIR + 'bert_baseline/',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='roc_auc_macro',
    greater_is_better=True,
    logging_steps=100,
    seed=42,
    fp16=torch.cuda.is_available(),
    report_to='none',
)

trainer_bert = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics_hf,
)

print("Fine-tuning BERT...")
t0 = time.time()
trainer_bert.train()
bert_train_time = time.time() - t0
print(f"Entrainement termine en {bert_train_time:.1f}s")

In [ ]:
# Evaluation BERT sur test set
t0 = time.time()
predictions = trainer_bert.predict(test_dataset)
bert_inference_time = time.time() - t0

logits = predictions.predictions
y_prob_bert = torch.sigmoid(torch.tensor(logits)).numpy()
y_pred_bert = (y_prob_bert > 0.5).astype(int)

results_bert = evaluate_model(
    y_test, y_pred_bert, y_prob_bert,
    'BERT (bert-base-uncased)', bert_train_time, bert_inference_time
)
all_results.append(results_bert)

# Sauvegarde du modele
trainer_bert.save_model(MODELS_DIR + 'bert_baseline/best')
tokenizer_bert.save_pretrained(MODELS_DIR + 'bert_baseline/best')
print(f"\nModele sauvegarde dans {MODELS_DIR}bert_baseline/best")

## 8. Sauvegarde des resultats et TF-IDF

In [ ]:
# Sauvegarder le vectorizer TF-IDF pour le dashboard
joblib.dump(tfidf, MODELS_DIR + 'tfidf_vectorizer.joblib')

# Sauvegarder toutes les metriques
with open(METRICS_DIR + 'baselines_metrics.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print("Metriques sauvegardees dans", METRICS_DIR + 'baselines_metrics.json')

In [ ]:
# Resume comparatif des baselines
results_df = pd.DataFrame(all_results)
cols = ['model', 'roc_auc_macro', 'f1_macro', 'precision_macro', 'recall_macro',
        'hamming_loss', 'train_time_sec', 'inference_time_sec']
print("\n" + "=" * 80)
print("RESUME DES BASELINES")
print("=" * 80)
print(results_df[cols].to_string(index=False))
print("\nProchaine etape : notebook 02_transformers.ipynb (modeles recents)")